# 02. 외부 데이터 EDA: AI Hub 경구약제 조합 이미지

[AI Hub 경구약제 이미지 데이터](https://www.aihub.or.kr/aihubdata/data/view.do?currMenu=115&topMenu=100&aihubDataSe=data&dataSetSn=576)(dataSetSn=576) 중
**조합 데이터만**, 그중 **`TL_2`와 그 원천 `TS_2`를 뺀** zip을 봅니다.
검증용(`VS_1`·`VL_1`)은 `VS_1`이 팀 Drive에 올라오기 전까지 `EXCLUDE`로 뺍니다. 대회 데이터 EDA는 `01_eda.ipynb`에 있습니다.

zip은 팀 Google Drive 공유 폴더(`aihub_pill`)에 있습니다. 노트북이 그 폴더를 받아서 풀고 읽으므로
**Colab에서는 위에서부터 실행만 하면 됩니다.** GPU는 필요 없습니다. 이미 받은 zip과 이미 푼 zip은 건너뜁니다.

| | zip 받는 곳 | 푸는 곳 |
|---|---|---|
| Colab | `data/external/aihub_zips` (런타임 디스크) | `data/external/aihub_combo` |
| 로컬 | 같음. `.env`에 `AIHUB_ZIP_DIR`이 있으면 받지 않고 그 폴더(예: Drive 데스크톱)를 씀 | 같음. `.env`의 `AIHUB_EXTRACT_DIR`로 바꿀 수 있음 |

- 폴더 링크는 `AIHUB_URL`로 받습니다. 저장소가 public이라 노트북에 적지 않습니다.
  환경 변수(`.env`, Colab은 Upload to Colab으로 올린 `/content/.env`) → Colab 보안 비밀 → 입력창 순서로 찾습니다.
- 폴더는 "링크가 있는 모든 사용자"로 공유되어 있어야 받을 수 있습니다.
- 용량: zip 약 24GB + 푼 파일 약 24GB. Colab 기본 디스크로 충분합니다.
- AI Hub는 해외 IP 다운로드를 막아 Colab에서 AI Hub로부터 직접 받을 수 없습니다. Drive 폴더는 국내 PC에서 채웠습니다.
- 포함·제외 기준: 경로에 `조합`이 들어간 zip 중 이름 앞 구분자(`TL_2_조합.zip` → `TL_2`)가 `EXCLUDE`에 없는 것.
  `TS_*`·`VS_*`는 원천(이미지), `TL_*`·`VL_*`는 라벨(JSON)입니다.

In [ ]:
import importlib.util
import os
import subprocess
import sys
from getpass import getpass
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None

BRANCH = "main"  # Colab에서 받을 저장소 브랜치
# 뺄 zip 구분자. TL_2 라벨을 안 쓰므로 그 이미지(TS_2)도 뺀다.
# 검증용(VL_1·VS_1)은 VS_1이 Drive에 올라오면 지워서 함께 본다.
EXCLUDE = ["TL_2", "TS_2", "VL_1", "VS_1"]
OVERWRITE = False  # True면 이미 푼 zip도 지우고 다시 푼다
SEED = 42

## 0. 환경 준비

Colab이면 저장소를 받아 import 경로에 넣고, 로컬이면 프로젝트 루트를 찾아 넣습니다. 그다음 `.env`를 읽습니다.
Colab에서는 `src.config`(torch·ultralytics·wandb)를 쓰지 않으므로 패키지를 따로 설치하지 않습니다.

In [ ]:
if IN_COLAB:
    ROOT = Path("/content/pill-bbox")
    git = ["git", "-C", str(ROOT)]
    if ROOT.exists():
        subprocess.run([*git, "fetch", "-q", "origin", BRANCH], check=True)
        subprocess.run([*git, "checkout", "-q", BRANCH], check=True)
        subprocess.run([*git, "pull", "-q"], check=True)
    else:
        url = "https://github.com/def-uk/pill-bbox.git"
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, str(ROOT)], check=True)
    env_file = Path("/content/.env")  # VS Code에서 Upload to Colab으로 올린 .env
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
    env_file = ROOT / ".env"

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
try:
    from dotenv import load_dotenv

    load_dotenv(env_file)  # 이미 설정된 환경 변수가 우선한다
except ImportError:
    pass

# 로컬은 AIHUB_ZIP_DIR(이미 있는 zip 폴더)·AIHUB_EXTRACT_DIR로 바꿀 수 있다.
# Colab은 올린 .env에 로컬 경로가 있어도 쓰지 않고 런타임 디스크에 받는다.
LOCAL_ZIP_DIR = None if IN_COLAB else os.getenv("AIHUB_ZIP_DIR")
ZIP_DIR = Path(LOCAL_ZIP_DIR or ROOT / "data" / "external" / "aihub_zips").expanduser()
extract_env = None if IN_COLAB else os.getenv("AIHUB_EXTRACT_DIR")
EXTRACT_DIR = Path(extract_env or ROOT / "data" / "external" / "aihub_combo").expanduser()
print(f"Colab: {IN_COLAB}\nzip 폴더: {ZIP_DIR}\n푸는 폴더: {EXTRACT_DIR}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from data_process.aihub_data import (
    download_drive_folder,
    extract_zips,
    find_combo_zips,
    list_combo_images,
    load_combo_annotations,
    zip_key,
)
from data_process.data_processing import build_cat_df, build_img_df, list_images, load_annotations
from visualization import draw_bboxes, image_grid, set_korean_font, show_image

np.random.seed(SEED)
pd.set_option("display.max_columns", 60)
print("한글 폰트:", set_korean_font())

## 1. zip 받기와 압축 풀기

팀 Drive 폴더에서 zip을 받고(Colab에서 수 분) 풉니다(수십 분). 이미 받은 zip은 건너뛰고, 끊긴 파일은 이어 받습니다.
다 푼 zip에는 표시 파일이 남아 다음 실행에서 건너뜁니다. Colab 런타임이 새로 뜨면 런타임 디스크가 비므로 다시 받고 풉니다.

In [ ]:
def env_value(name: str) -> str:
    """환경 변수 → Colab 보안 비밀 → 입력창 순서로 값을 찾는다."""
    if os.getenv(name):
        return os.environ[name]
    if IN_COLAB:
        from google.colab import userdata

        try:
            return userdata.get(name)
        except Exception:  # 보안 비밀이 없거나 이 노트북의 접근을 허용하지 않음
            pass
    return getpass(f"{name}: ")


if LOCAL_ZIP_DIR:
    print(f"받지 않고 AIHUB_ZIP_DIR을 씁니다: {ZIP_DIR}")
else:
    download_drive_folder(env_value("AIHUB_URL"), ZIP_DIR)

In [ ]:
zip_paths = find_combo_zips(ZIP_DIR, exclude=EXCLUDE)
zip_df = pd.DataFrame(
    {
        "zip": [p.name for p in zip_paths],
        "구분자": [zip_key(p) for p in zip_paths],
        "크기(GB)": [round(p.stat().st_size / 1e9, 2) for p in zip_paths],
    }
)
print(f"사용할 조합 zip: {len(zip_df)}개, 합계 {zip_df['크기(GB)'].sum():.1f}GB (제외: {EXCLUDE})")
display(zip_df)
if zip_df.empty:
    raise FileNotFoundError(f"{ZIP_DIR} 아래에 경로에 '조합'이 들어간 zip이 없습니다.")

In [ ]:
extracted_dirs = extract_zips(zip_paths, EXTRACT_DIR, overwrite=OVERWRITE)

## 2. 데이터 로드

대회 데이터와 같은 로더(`data_process/data_processing.py`)를 씁니다. 표 구성도 `01_eda.ipynb`와 같습니다.

| 표 | 한 행 | 설명 |
|---|---|---|
| `ann_df` | bbox 1개 | 이미지 정보 + 약 정보 + bbox(COCO `[x, y, w, h]`). `zip_name`은 읽어 온 zip |
| `image_index` | 디스크의 이미지 1장 | `file_name`, `zip_name`, `path` |
| `img_df` | 이미지 1장 | 라벨 없는 이미지도 포함(`n_boxes=0`). `combo`는 파일 이름의 알약 조합 |
| `cat_df` | 클래스 1개 | `category_id` ↔ 약 이름, bbox 수, 이미지 수 |
| `load_error_df` | 오류 1건 | JSON 읽기·연결 실패 |

라벨 없는 이미지(`n_boxes=0`)가 있으면 원천·라벨 zip 짝이 맞는지(`EXCLUDE`) 확인합니다.

AI Hub 원본의 `category_id`는 모두 1(`Drug`)입니다. 로더가 대회 데이터처럼 `drug_N`의 숫자(K-001900 → 1900)로 바꾸고 `category_name`은 `dl_name`으로 둡니다. 원래 값은 `category_id_raw`에 있습니다.
`dl_idx`는 약 코드보다 1 작은 값이라 클래스로 쓰지 않습니다.

In [ ]:
ann_df, load_error_df = load_combo_annotations(extracted_dirs)
image_index = list_combo_images(extracted_dirs)
if image_index.empty:
    raise FileNotFoundError(
        "이미지가 없습니다. 원천 zip(TS·VS)이 팀 Drive에 있는지, EXCLUDE에 빠졌는지 확인하세요."
    )
img_df = build_img_df(ann_df, list(image_index["path"])).merge(
    image_index[["file_name", "zip_name"]].drop_duplicates("file_name"), on="file_name"
)
cat_df = build_cat_df(ann_df)
img_path = dict(zip(image_index["file_name"], image_index["path"], strict=True))

n_dup = image_index["file_name"].duplicated().sum()
print(f"bbox: {len(ann_df)}개, JSON: {ann_df['source_json'].nunique()}개, 클래스: {len(cat_df)}개")
print(f"이미지: {len(img_df)}장 (라벨 없음: {(img_df['n_boxes'] == 0).sum()}장)")
print(f"이름이 겹치는 이미지 파일: {n_dup}개")
print(f"JSON이 가리키지만 디스크에 없는 이미지: {len(set(ann_df['file_name']) - set(img_path))}장")
print(f"로드 오류: {len(load_error_df)}건")
if not load_error_df.empty:
    display(load_error_df.head(20))
display(ann_df.head(3))

## 3. 규모와 구조

In [ ]:
# zip별 파일 수. 원천 zip은 이미지, 라벨 zip은 JSON·bbox만 있다.
per_zip = (
    pd.DataFrame(
        {
            "이미지": image_index.groupby("zip_name").size(),
            "JSON": ann_df.groupby("zip_name")["source_json"].nunique(),
            "bbox": ann_df.groupby("zip_name").size(),
        }
    )
    .fillna(0)
    .astype(int)
)
display(per_zip)

print("JSON에 적힌 해상도 (width × height: bbox 수)")
display(ann_df.groupby(["width", "height"]).size().rename("bbox 수").to_frame())

In [ ]:
# 이미지당 bbox 수와 파일 이름에 적힌 약 코드 수. 둘이 다르면 라벨 누락 후보다.
print("파일 이름의 약 코드 수 × bbox 수 (값: 이미지 수)")
display(
    pd.crosstab(img_df["n_codes_in_name"], img_df["n_boxes"]).rename_axis(
        index="이름의 약 코드 수", columns="bbox 수"
    )
)
print(f"조합 수: {img_df['combo'].nunique()}, 조합당 이미지 수:")
display(img_df.groupby("combo").size().describe().to_frame("이미지 수").T)

In [ ]:
print("클래스별 bbox 수")
display(cat_df["n_boxes"].describe().to_frame().T)
display(cat_df.sort_values("n_boxes", ascending=False).head(10))
display(cat_df.sort_values("n_boxes").head(10))

## 4. 대회 데이터와 비교

대회 데이터(`data/raw`)가 있을 때만 실행합니다. 대회 이미지가 AI Hub 조합 데이터에서 나왔을 수 있으므로,
같은 파일 이름이 대회 **test**에 있으면 그대로 학습에 넣으면 안 됩니다(누수). 이름만 비교하므로
같은 사진인지는 해시나 눈으로 다시 확인합니다.

In [ ]:
raw_dir = ROOT / "data" / "raw"
if (raw_dir / "sprint_ai_project1_data").is_dir():
    raw_dir = raw_dir / "sprint_ai_project1_data"

if (raw_dir / "train_annotations").is_dir():
    comp_ann_df, _ = load_annotations(raw_dir / "train_annotations")
    comp_files = {
        split: {p.name for p in list_images(raw_dir / f"{split}_images")}
        for split in ["train", "test"]
    }
    aihub_files = set(img_df["file_name"])
    for split, files in comp_files.items():
        n_same = len(files & aihub_files)
        print(f"대회 {split} 이미지 {len(files)}장 중 AI Hub에 같은 이름: {n_same}장")

    comp_cats = set(comp_ann_df["category_id"])
    aihub_cats = set(cat_df["category_id"])
    print(
        f"클래스: 대회 {len(comp_cats)}종, AI Hub {len(aihub_cats)}종, "
        f"겹침 {len(comp_cats & aihub_cats)}종, 대회에만 있음 {len(comp_cats - aihub_cats)}종"
    )
    overlap_cat_df = cat_df[cat_df["category_id"].isin(comp_cats)]
    print("대회 클래스의 AI Hub bbox 수")
    display(overlap_cat_df["n_boxes"].describe().to_frame().T)
else:
    print(f"대회 데이터가 없어 건너뜁니다: {raw_dir}")

## 5. 예시 이미지

In [ ]:
sample = img_df[img_df["n_boxes"] > 0].sample(
    min(8, (img_df["n_boxes"] > 0).sum()), random_state=SEED
)
fig, axes = image_grid(len(sample), cols=4)
for ax, file_name in zip(axes, sample["file_name"], strict=True):
    show_image(ax, img_path[file_name], title=file_name, fontsize=6)
    draw_bboxes(ax, ann_df[ann_df["file_name"] == file_name], label_col="dl_name")
fig.tight_layout()
plt.show()

## 6. 다음 분석 거리

- 클래스 분포: 대회 56종과 겹치는 클래스의 이미지·bbox 수, 대회에 없는 클래스를 쓸지
- bbox: 크기·종횡비·이미지 밖 좌표(`bbox_flags`), 대회 데이터와 분포 비교
- 촬영 조건: `camera_la`, `back_color`, `light_color`, `drug_dir` 분포가 대회 데이터와 같은지
- 라벨 품질: 이름의 약 코드 수와 bbox 수가 다른 이미지(위 3장 표), 빈 bbox
- 누수: 대회 test와 같은 사진(이름·해시) 제외 기준
- 학습에 넣을 때: `category_id` → 대회 클래스 번호 매핑, 라벨 없는 이미지는 빼기(배경으로 학습되지 않게)